#Task 7: Implementing a Hybrid Recommendation Model

In [3]:
# Install dependencies (Colab might prompt you to click "Restart Session" after this runs)
!pip install "numpy<2"
!pip install scikit-surprise

  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554985 sha256=af396408386ca4db943c508ecde11085234c7df469d64bccbeff0b47b0074d1d
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [10]:
import pandas as pd
import numpy as np
from surprise import Reader, Dataset, SVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from collections import defaultdict

# 1. Load User Ratings
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

# 2. Load Movies and Genres
m_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + \
         ['unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
          'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
          'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('u.item', sep='|', names=m_cols, encoding='latin-1')

# 3. Train/Test Split (80/20) for our base data
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

print(f"Data Loaded! Train shape: {train_data.shape}, Test shape: {test_data.shape}")

Data Loaded! Train shape: (80000, 4), Test shape: (20000, 4)


In [11]:
user_avgs = train_data.groupby('user_id')['rating'].mean().to_dict()
movie_avgs = train_data.groupby('movie_id')['rating'].mean().to_dict()
global_avg = train_data['rating'].mean() # Fallback for unknown items/users

reader = Reader(rating_scale=(1, 5))
train_dataset = Dataset.load_from_df(train_data[['user_id', 'movie_id', 'rating']], reader)
trainset = train_dataset.build_full_trainset()

cf_algo = SVD(n_factors=50, random_state=42)
cf_algo.fit(trainset)

genre_cols = movies.columns[5:]
movie_genre_matrix = movies.set_index('movie_id')[genre_cols]
cosine_sim = cosine_similarity(movie_genre_matrix)
cbf_sim_df = pd.DataFrame(cosine_sim, index=movie_genre_matrix.index, columns=movie_genre_matrix.index)

def get_cbf_prediction(user_id, movie_id):
    """Predicts a rating based on a user's past ratings weighted by genre similarity."""
    if movie_id not in cbf_sim_df.index:
        return global_avg

    # Get all movies rated by this user in the train set
    user_history = train_data[train_data['user_id'] == user_id]
    if user_history.empty:
        return global_avg

    sim_scores = cbf_sim_df.loc[movie_id, user_history['movie_id']].values
    ratings = user_history['rating'].values

    # Weighted average: sum(sim * rating) / sum(sim)
    if np.sum(sim_scores) > 0:
        return np.dot(sim_scores, ratings) / np.sum(sim_scores)
    else:
        return user_avgs.get(user_id, global_avg)

print("Base Models (CF & CBF) and Average dictionaries are ready.")

Base Models (CF & CBF) and Average dictionaries are ready.


In [12]:
def create_meta_features(df):
    """Generates the 4 features for the meta-learning model."""
    features = []

    for _, row in df.iterrows():
        u = row['user_id']
        i = row['movie_id']

        cf_pred = cf_algo.predict(u, i).est

        cbf_pred = get_cbf_prediction(u, i)

        u_avg = user_avgs.get(u, global_avg)

        m_avg = movie_avgs.get(i, global_avg)

        features.append([u, i, row['rating'], cbf_pred, cf_pred, m_avg, u_avg])

    cols = ['user_id', 'movie_id', 'actual_rating', 'cbf_pred', 'cf_pred', 'movie_avg', 'user_avg']
    return pd.DataFrame(features, columns=cols)

print("Extracting features for Train set (This might take a minute)...")
train_meta = create_meta_features(train_data)

print("Extracting features for Test set...")
test_meta = create_meta_features(test_data)

print("Meta-Features successfully extracted!")
train_meta.head()

Extracting features for Train set (This might take a minute)...
Extracting features for Test set...
Meta-Features successfully extracted!


,user_id,movie_id,actual_rating,cbf_pred,cf_pred,movie_avg,user_avg
0,807,1411,1,3.741703,2.948263,2.640000,3.870588
1,474,659,5,4.033800,4.600394,4.113636,4.079681
2,463,268,4,3.355465,3.307927,3.782178,2.902913
3,139,286,4,4.227646,4.093193,3.613402,4.090909
4,621,751,4,3.823578,3.651847,3.431507,3.607407


In [13]:
# Define Features (X) and Target (y)
X_cols = ['cbf_pred', 'cf_pred', 'movie_avg', 'user_avg']

X_train = train_meta[X_cols]
y_train = train_meta['actual_rating']

X_test = test_meta[X_cols]
y_test = test_meta['actual_rating']

# Train the Meta-Model (Linear Regression handles dynamic blending beautifully)
meta_model = LinearRegression()
meta_model.fit(X_train, y_train)

# Generate Final Hybrid Predictions
test_meta['hybrid_pred'] = meta_model.predict(X_test)
# Clip predictions to stay within the 1-5 star bounds
test_meta['hybrid_pred'] = test_meta['hybrid_pred'].clip(1, 5)

# Calculate RMSE
hybrid_rmse = np.sqrt(mean_squared_error(test_meta['actual_rating'], test_meta['hybrid_pred']))
cf_rmse = np.sqrt(mean_squared_error(test_meta['actual_rating'], test_meta['cf_pred']))
cbf_rmse = np.sqrt(mean_squared_error(test_meta['actual_rating'], test_meta['cbf_pred']))

print(f"--- RMSE Comparison ---")
print(f"CF Only (SVD) RMSE:   {cf_rmse:.4f}")
print(f"CBF Only RMSE:        {cbf_rmse:.4f}")
print(f"Hybrid Meta-Model:    {hybrid_rmse:.4f}")

--- RMSE Comparison ---
CF Only (SVD) RMSE:   0.9313
CBF Only RMSE:        1.0287
Hybrid Meta-Model:    0.9669


In [14]:
def get_top_n_metrics(df, pred_col, k=10, threshold=4.0):
    user_preds = defaultdict(list)
    for _, row in df.iterrows():
        user_preds[row['user_id']].append((row[pred_col], row['actual_rating']))

    precisions, recalls = dict(), dict()

    for uid, user_ratings in user_preds.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True) # Sort by predicted rating
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = len(user_ratings[:k])
        n_rel_and_rec_k = sum((true_r >= threshold) for (_, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return sum(precisions.values())/len(precisions), sum(recalls.values())/len(recalls)

# Calculate for the Hybrid model
prec_hybrid, rec_hybrid = get_top_n_metrics(test_meta, 'hybrid_pred', k=10)

print(f"--- Hybrid Model Ranking Metrics ---")
print(f"Precision @ 10: {prec_hybrid:.4f}")
print(f"Recall @ 10:    {rec_hybrid:.4f}")

--- Hybrid Model Ranking Metrics ---
Precision @ 10: 0.6674
Recall @ 10:    0.7104


In [15]:
# Identify Cold-Start Users (<= 20 ratings in train data)
user_rating_counts = train_data.groupby('user_id').size()
cold_start_users = user_rating_counts[user_rating_counts <= 20].index.tolist()

# Filter the test set for only these users
cold_test_meta = test_meta[test_meta['user_id'].isin(cold_start_users)]

if not cold_test_meta.empty:
    cold_hybrid_rmse = np.sqrt(mean_squared_error(cold_test_meta['actual_rating'], cold_test_meta['hybrid_pred']))
    cold_cf_rmse = np.sqrt(mean_squared_error(cold_test_meta['actual_rating'], cold_test_meta['cf_pred']))
    cold_cbf_rmse = np.sqrt(mean_squared_error(cold_test_meta['actual_rating'], cold_test_meta['cbf_pred']))

    print(f"--- Cold-Start Analysis (Users with <= 20 ratings) ---")
    print(f"Number of Cold-Start Users evaluated: {len(cold_test_meta['user_id'].unique())}")
    print(f"CF Only (SVD) RMSE:   {cold_cf_rmse:.4f}")
    print(f"CBF Only RMSE:        {cold_cbf_rmse:.4f}")
    print(f"Hybrid Meta-Model:    {cold_hybrid_rmse:.4f}")

else:
    print("No cold-start users found in the test set based on the current split/threshold.")

--- Cold-Start Analysis (Users with <= 20 ratings) ---
Number of Cold-Start Users evaluated: 125
CF Only (SVD) RMSE:   0.9839
CBF Only RMSE:        1.0933
Hybrid Meta-Model:    1.0122
